<a href="https://colab.research.google.com/github/neelameghana192311002/CSA6301---Threat-Intelligence-and-Network-Security/blob/main/UNIT4LAB/Exercise_9_Defense_in_Depth_Layered_Control_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def simulate_defense(attack, layers):
    """
    layers: ordered list of (layer_name, catch_function).

    catch_function receives the attack dictionary and returns True
    if that layer stops the attack.

    Returns the name of the first layer that catches the attack,
    or "BREACH" if it passes through every layer undetected.
    """
    for layer_name, catch_fn in layers:
        if catch_fn(attack):
            return layer_name

    return "BREACH"


def spam_filter_catches(attack):
    return attack.get("known_phishing_domain", False)


def endpoint_av_catches(attack):
    return attack.get("known_malware_hash", False)


def segmentation_catches(attack):
    return (
        attack.get("attempts_lateral_movement", False)
        and not attack.get("segmentation_bypassed", False)
    )


def siem_monitoring_catches(attack):
    return attack.get("generates_anomalous_traffic", False)

In [2]:
def test_experiment9():
    layers = [
        ("Email Spam Filter", spam_filter_catches),
        ("Endpoint Antivirus", endpoint_av_catches),
        ("Network Segmentation", segmentation_catches),
        ("SIEM Monitoring", siem_monitoring_catches),
    ]

    # Phishing email evades the spam filter (unknown domain),
    # but AV catches the attachment
    attack1 = {
        "known_phishing_domain": False,
        "known_malware_hash": True,
    }

    assert simulate_defense(attack1, layers) == "Endpoint Antivirus"

    # Evades spam filter and AV, but is caught while attempting lateral movement
    attack2 = {
        "known_phishing_domain": False,
        "known_malware_hash": False,
        "attempts_lateral_movement": True,
        "segmentation_bypassed": False,
    }

    assert simulate_defense(attack2, layers) == "Network Segmentation"

    # Evades every layer entirely
    attack3 = {
        "known_phishing_domain": False,
        "known_malware_hash": False,
        "attempts_lateral_movement": False,
        "generates_anomalous_traffic": False,
    }

    assert simulate_defense(attack3, layers) == "BREACH"

    print("All test cases passed.")


test_experiment9()

All test cases passed.
